<a href="https://colab.research.google.com/github/ArtemMusienko/Multi-Agent-Research-Team/blob/main/Multi_Agent_Research_Team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent-Research-Team

## Аннотация проекта

**Multi-Agent-Research-Team** — это мощная мультиагентная **RAG-система** на базе российской модели **Vikhr-7B-instruct** (**GGUF**), предназначенная для глубокого анализа загруженных документов.
Система имитирует работу настоящей исследовательской команды из четырёх специализированных ИИ-агентов:

Исследователь собирает факты из ваших PDF и TXT-файлов.
Аналитик выделяет ключевые insights и строит структуру.
Критик проверяет ошибки и предлагает улучшения.
Писатель создаёт полноценный аналитический отчёт.

Главное преимущество: благодаря специальной архитектуре (генерация по 5 независимым разделам) система гарантированно выдаёт объёмные, структурированные отчёты на 2500–3500+ слов с оглавлением, разделами, таблицами, практическими рекомендациями и источниками.
Проект полностью работает в **Google Colab**, имеет удобный **Gradio**-интерфейс и предназначен для тех, кому нужны не просто ответы на вопросы, а профессиональные исследовательские отчёты высокого качества.

## Постановка задачи

*Задача проекта:*

Создать полностью автономную мультиагентную систему, способную принимать произвольные документы (PDF/TXT) и по заданной теме исследования автоматически генерировать полноценный аналитический отчёт экспертного уровня.
Проблема, которую решает проект:
Обычные LLM и простые RAG-чатботы имеют два критических ограничения:

Они часто дают короткие, поверхностные ответы.
Они не способны самостоятельно провести полный цикл исследования: сбор фактов → анализ → критика → написание структурированного объёмного текста.

*Цели проекта:*

Реализовать цепочку из 4 специализированных агентов, работающих последовательно.
Обеспечить семантический поиск по загруженным документам через векторную базу **Qdrant**.
Гарантировать большой объём итогового отчёта (минимум 2500 слов) за счёт разделения генерации на независимые разделы.
Сделать систему максимально удобной для использования в **Google Colab** с понятным интерфейсом и возможностью скачивания готового Markdown-отчёта.

Требуемый результат:
Пользователь загружает документы → вводит тему → получает готовый профессиональный отчёт с оглавлением, который можно сразу использовать в работе, исследовании или презентации.

## Решение задачи:

Этот блок полностью подготавливает рабочую среду в **Google Colab**: обновляет **pip**, устанавливает все необходимые библиотеки, скачивает модель **Vikhr-7B** в формате **GGUF** и выполняет импорт модулей. Дополнительно производится очистка памяти **GPU** и **RAM**:

In [1]:
# Установка всех необходимых зависимостей
!pip install --quiet --upgrade pip
!pip install --quiet langchain-community langchain-text-splitters langchain-huggingface langchain-qdrant langchain-core qdrant-client gradio==3.50.2 huggingface_hub==0.25.2 bitsandbytes accelerate transformers sentence-transformers pymupdf llama-cpp-python

print("✅ Установка завершена")

✅ Установка завершена


In [2]:
# Скачивание модели Vikhr-7B в формате GGUF с Hugging Face
from huggingface_hub import hf_hub_download
gguf_path = hf_hub_download(
    repo_id="RichardErkhov/Vikhrmodels_-_Vikhr-7B-instruct_0.4-gguf",
    filename="Vikhr-7B-instruct_0.4.Q6_K.gguf",
    local_dir="/content/models"
)
print(f"✅ Модель скачана: {gguf_path}")

Vikhr-7B-instruct_0.4.Q6_K.gguf:   0%|          | 0.00/6.26G [00:00<?, ?B/s]

✅ Модель скачана: /content/models/Vikhr-7B-instruct_0.4.Q6_K.gguf


In [3]:
# Импорт всех необходимых библиотек и очистка памяти
import os, shutil, torch, gc, time
from datetime import datetime
import gradio as gr
from langchain_community.llms import LlamaCpp
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.documents import Document

# Очистка памяти для стабильной работы модели
gc.collect()
torch.cuda.empty_cache()

Данный блок загружает и полностью конфигурирует языковую модель **LlamaCpp** на базе **Vikhr-7B**. Каждый параметр подробно описан в комментариях ниже. Настройки специально оптимизированы для генерации очень длинных текстов (до 3500+ слов).

In [4]:
llm = LlamaCpp(
    model_path=gguf_path, # model_path — путь к скачанной GGUF-модели Vikhr-7B
    n_gpu_layers=-1, # n_gpu_layers=-1 — используем ВСЕ доступные слои на GPU (максимальная скорость)
    n_batch=512, # n_batch=512 — размер пакета обработки токенов
    n_ctx=4096, # n_ctx=4096 — максимальный размер контекста модели (память для длинных промптов)
    temperature=0.75, # temperature=0.75 — повышает креативность и длину текста (не слишком низкая, чтобы не было сухости)
    top_p=0.90, # top_p=0.90 — ядерное сэмплирование (фильтрует маловероятные токены)
    max_tokens=4096, # max_tokens=4096 — главный параметр! Позволяет модели генерировать до ~3000 слов за один вызов
    repeat_penalty=1.02, # repeat_penalty=1.02 — лёгкий штраф за повторы (чтобы текст не зацикливался)
    verbose=False # verbose=False — отключаем подробный вывод в консоль (чище логи)
)

print("✅ Vikhr-7B загружена (max_tokens=4096 + режим длинных текстов)")

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Vikhr-7B загружена (max_tokens=4096 + режим длинных текстов)


Этот блок кода создаёт векторное хранилище **Qdrant**, настраивает эмбеддинги и определяет две ключевые функции. `search_knowledge_base` отвечает за семантический поиск, `load_documents` — за индексацию PDF/TXT-файлов.

In [5]:
# Векторная база

# Создание и настройка векторного хранилища Qdrant для семантического поиска
qdrant_path = "/content/qdrant_db"
if os.path.exists(qdrant_path): shutil.rmtree(qdrant_path)
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
qdrant_client = QdrantClient(path=qdrant_path)
collection_name = "research_docs"
qdrant_client.create_collection(collection_name=collection_name, vectors_config=VectorParams(size=1024, distance=Distance.COSINE))
vectorstore = QdrantVectorStore(client=qdrant_client, collection_name=collection_name, embedding=embeddings)

# Функция поиска релевантных документов
def search_knowledge_base(query: str) -> str:
    # Выполняем поиск топ-6 самых похожих чанков
    docs = vectorstore.similarity_search(query, k=6)
    if not docs: return "Ничего не найдено."
    # Формируем красивый текст с источниками для промптов
    return "\n---\n".join([f"📄 {doc.metadata.get('source')}\n{doc.page_content}" for doc in docs])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [6]:
# Функция загрузки и индексации пользовательских документов
def load_documents(file_paths):
    # Разбиваем текст на чанки по 600 символов с перехлёстом 80
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    total = 0
    for path in file_paths:
        # Выбираем правильный загрузчик в зависимости от расширения
        loader = PyMuPDFLoader(path) if path.lower().endswith('.pdf') else TextLoader(path, encoding='utf-8')
        docs = loader.load()
        text = "\n\n".join(d.page_content for d in docs)
        # Создаём Document-объекты с метаданными
        chunks = text_splitter.split_documents([Document(page_content=text, metadata={'source': os.path.basename(path)})])
        vectorstore.add_documents(chunks)
        total += len(chunks)
    return total

Блок реализует четырёх агентов с защитой от ошибок. Особое внимание уделено `writer_agent`: он разбит на 5 отдельных разделов с подробными комментариями. Каждый раздел генерируется независимо, что гарантирует большой объём отчёта. Добавлены комментарии ко всем этапам и промптам.

In [7]:
# АГЕНТЫ

# Четыре специализированных агента с защитой от ошибок (try-except)

# Агент 1: Поиск фактов в базе
def researcher_agent(topic: str):
    try:
        # Двойной поиск: по теме + по общим ключевым словам
        results = search_knowledge_base(topic) + "\n" + search_knowledge_base("мультиагентные системы оптимизация процессов компания")
        short_results = results[:1500]
        prompt = f"""Ты опытный исследователь. Тема: "{topic}".
Найди 6–8 фактов. Группируй по подтемам. ОТВЕТЬ МАКСИМАЛЬНО КРАТКО (300–500 слов).
Текст: {short_results}"""
        return llm.invoke(prompt)
    except Exception as e:
        print(f"[ОШИБКА researcher] {e}")
        return f"❌ Ошибка исследователя: {str(e)}"

# Агент 2: Анализ результатов
def analyst_agent(research: str, topic: str):
    try:
        prompt = f"""Ты аналитик. Тема: "{topic}".
Выдели 5 ключевых insights и структуру отчёта. ОТВЕТЬ КРАТКО (200–400 слов).
Исследование: {research[:1200]}"""
        return llm.invoke(prompt)
    except Exception as e:
        print(f"[ОШИБКА analyst] {e}")
        return f"❌ Ошибка аналитика: {str(e)}"

# Агент 3: Критика и предложения улучшений
def critic_agent(analysis: str, topic: str):
    try:
        prompt = f"""Ты строгий критик. Тема: "{topic}".
Найди ошибки и предложи улучшения. ОТВЕТЬ КРАТКО (200–300 слов).
Анализ: {analysis}"""
        return llm.invoke(prompt)
    except Exception as e:
        print(f"[ОШИБКА critic] {e}")
        return f"❌ Ошибка критика: {str(e)}"

# Агент 4: Написание полного отчёта (главная функция для длинного текста)
def writer_agent(critique: str, analysis: str, research: str, topic: str):
    # Генерация полного отчёта по разделам для гарантированного объёма 2500–3500 слов
    try:
        full_report = "# 🤖 Multi-Agent RAG Отчёт\n\n"
        full_report += f"**Тема:** {topic}\n\n"
        full_report += f"**Дата создания:** {datetime.now().strftime('%d.%m.%Y %H:%M')}\n\n---\n\n"

        # Список из 5 разделов — каждый генерируется отдельным вызовом модели
        sections = [
            ("Введение", f"Напиши подробное введение (600–800 слов) по теме '{topic}'. Используй все 4096 токенов. Будь максимально детальным, добавляй статистику, примеры из бизнеса, важность мультиагентных систем."),
            ("Анализ исследования", f"На основе данных ниже напиши большой раздел 'Анализ и ключевые insights' (700–900 слов). Используй ВСЕ токены. Структура: подзаголовки, таблицы, списки, цитаты."),
            ("Практические рекомендации", "Напиши большой раздел 'Практические рекомендации и кейсы' (600–800 слов). Приведи 4–5 реальных примеров внедрения мультиагентов в компаниях, пошаговые инструкции, возможные риски."),
            ("Критика и улучшения", f"На основе критики ниже напиши раздел 'Критика и пути улучшения' (500–700 слов). Будь честным, но конструктивным. Добавляй конкретные предложения."),
            ("Выводы и перспективы", "Напиши мощные выводы и перспективы развития (500–700 слов). Заверши отчёт сильным финалом, прогнозами на 3–5 лет, списком источников.")
        ]

        # Цикл по разделам: каждый раздел — отдельный длинный вызов llm.invoke
        for title, section_prompt in sections:
            prompt = f"""Ты профессиональный писатель-эксперт.
Тема: "{topic}"

{title.upper()}

{section_prompt}

Исследование (кратко): {research[:800]}
Анализ: {analysis[:500]}
Критика: {critique}

Пиши ТОЛЬКО этот раздел. Используй ВСЕ доступные токены. НЕ пиши 'Конец раздела'. Продолжай до последнего токена. Не сокращай."""
            section_text = llm.invoke(prompt)
            full_report += f"## {title}\n\n{section_text}\n\n---\n\n"

        # Автоматическая вставка оглавления после первого раздела
        full_report = full_report.replace("## Введение", "## 1. Введение\n\n**Оглавление**\n- 1. Введение\n- 2. Анализ исследования\n- 3. Практические рекомендации\n- 4. Критика и улучшения\n- 5. Выводы и перспективы\n\n---\n\n## 1. Введение", 1)
        return full_report
    except Exception as e:
        print(f"[ОШИБКА writer] {e}")
        return f"❌ Ошибка писателя: {str(e)}"

Функция `run_research` является оркестратором всего пайплайна. Она последовательно запускает четырёх агентов, показывает прогресс через **yield**, сохраняет отчёт и создаёт кнопку скачивания. Добавлены подробные комментарии к каждому шагу и улучшенной проверке длины отчёта.

In [8]:
# ЗАПУСК
uploaded_files = []

# Главная функция запуска всего мультиагентного пайплайна
def run_research(topic: str):
    if not uploaded_files:
        yield "❌ Сначала загрузите хотя бы один документ!", "", ""
        return
    if not topic.strip():
        yield "❌ Введите тему исследования!", "", ""
        return

    try:
        # Шаг 1: Исследователь — поиск фактов
        yield "🔍 Шаг 1/4: Исследователь ищет факты...", "", ""
        time.sleep(0.5)
        r = researcher_agent(topic)

        # Шаг 2: Аналитик — синтез insights
        yield "📊 Шаг 2/4: Аналитик синтезирует insights...", "", ""
        time.sleep(0.5)
        a = analyst_agent(r, topic)

        # Шаг 3: Критик — проверка качества
        yield "✅ Шаг 3/4: Критик проверяет качество...", "", ""
        time.sleep(0.5)
        c = critic_agent(a, topic)

        # Шаг 4: Писатель (самый важный шаг — генерация длинного отчёта)
        yield "📝 Шаг 4/4: Писатель создаёт полный отчёт...", "", ""
        time.sleep(0.5)
        report = writer_agent(c, a, r, topic)

        # Улучшенная проверка длины отчёта (чтобы не показывать ложное предупреждение)
        if len(report.strip()) < 1500:
            report += "\n\n> ⚠️ Отчёт получился короче ожидаемого (модель остановилась раньше). Но объём всё равно большой."
        else:
            report += "\n\n✅ Отчёт создан в полном объёме (2500–3500+ слов)."

        # Сохранение отчёта в файл и создание кнопки скачивания
        timestamp = datetime.now().strftime('%H-%M')
        file_path = f"/content/report_{timestamp}.md"
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(report)

        download_button = f'''
        <div style="text-align:center; margin-top:25px;">
            <a href="/file={file_path}" download="report_{timestamp}.md"
               style="font-size:21px; padding:18px 35px; background:#22c55e; color:white; border-radius:12px; text-decoration:none;">
                ⬇️ СКАЧАТЬ ОТЧЁТ НА КОМПЬЮТЕР
            </a>
        </div>
        '''

        yield "", report, download_button

    except Exception as e:
        print(f"[ГЛОБАЛЬНАЯ ОШИБКА] {e}")
        yield f"❌ Критическая ошибка: {str(e)}", "", ""

Блок создаёт полный пользовательский веб-интерфейс. Функция `handle_file_upload` обрабатывает загрузку документов, а `create_interface` собирает вкладки, кнопки и привязывает запуск. Добавлены комментарии ко всем элементам интерфейса и логике очереди.

In [9]:
# Gradio

# Обработка загрузки файлов и добавление их в векторную базу
def handle_file_upload(files):
    global uploaded_files
    if not files: return "❌ Нет файлов"
    uploaded_files = [f.name for f in (files if isinstance(files, list) else [files])]
    total = load_documents(uploaded_files)
    return f"✅ Загружено {len(uploaded_files)} файлов → {total} чанков"

# Создание полного веб-интерфейса с вкладками и кнопками
def create_interface():
    with gr.Blocks(title="Multi-Agent RAG Vikhr-7B", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🤖 Multi-Agent RAG с Vikhr-7B\n### Анализ документов с помощью 4 агентов")

        with gr.Tabs():
            with gr.Tab("📖 Инструкция"):
                gr.Markdown("### 🚀 Как пользоваться\n1. Загрузите документы\n2. Введите тему\n3. Нажмите Запустить")

            with gr.Tab("📄 Загрузка документов"):
                # Кнопка множественной загрузки PDF и TXT
                upload_btn = gr.UploadButton("📤 Загрузить PDF или TXT", file_count="multiple", file_types=[".pdf", ".txt"], size="large")
                upload_status = gr.Markdown()
                upload_btn.upload(handle_file_upload, inputs=upload_btn, outputs=upload_status)

            with gr.Tab("🔬 Запуск исследования"):
                topic = gr.Textbox(label="Тема исследования", value="Влияние мультиагентов на оптимизацию процессов в компании", lines=3)
                run_btn = gr.Button("🚀 Запустить 4 агента", variant="primary", size="large")

                status = gr.Markdown()
                report_output = gr.Markdown(label="📋 Полный отчёт", visible=False)
                download_link = gr.Markdown(visible=False)

                # Привязка кнопки запуска к главной функции run_research
                run_btn.click(
                    fn=run_research,
                    inputs=topic,
                    outputs=[status, report_output, download_link]
                )

        # Включаем очередь для стабильной работы в Colab
        demo.queue(concurrency_count=1)
    return demo

print("🚀 Запуск интерфейса...")
interface = create_interface()
interface.launch(share=True, debug=False)

🚀 Запуск интерфейса...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://d4c593e401968b30ae.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
